# Chapter 3: Grouping, Merging, and Concatenating in Pandas

A beginner-friendly workbook packed with detailed explanations, visual examples, and practice exercises:
1. **3.1 Grouping Data (`groupby`)**: The Split-Apply-Combine pattern, inspecting group buckets, `.get_group()`, multi-column grouping, and custom rule-based grouping
2. **3.2 Concatenating Data (`pd.concat`)**: Stacking tables vertically (`axis=0`) and horizontally (`axis=1`), resetting indices (`ignore_index=True`), and column alignment (`join='outer'` vs `join='inner'`)
3. **3.3 Merging DataFrames (`df.merge`)**: Relational database joins (`inner`, `left`, `right`, `outer`), linking shared keys (`on`), and joining mismatched column headers (`left_on` / `right_on`)

---
---

## 3.1 Grouping Data with `groupby`

### Concept Summary
When analyzing real-world data, you rarely want to look at single rows one by one. Instead, you want to summarize numbers by category—such as calculating average revenue by store, total hours worked per department, or exam scores by school class.

Pandas uses the **Split-Apply-Combine** pattern:

```text
=========================================================================================
                      THE SPLIT - APPLY - COMBINE WORKFLOW
=========================================================================================

   1. SPLIT                     2. APPLY (e.g. Mean)            3. COMBINE

 [ Original Data ]            [ Buckets in Memory ]          [ Summary Table ]
 Category | Score              Group A:                       Category | Mean Score
 ---------+------   ------->  Row 0: 10   --> (10+30)/2 = 20  ---------+-----------
    A     |  10               Row 2: 30                       A        |   20.0
    B     |  20                                               B        |   30.0
    A     |  30               Group B:
    B     |  40               Row 1: 20   --> (20+40)/2 = 30
                              Row 3: 40
=========================================================================================
```

---

### Grouping Tools: Deep Dive with Examples

#### 1. Creating a GroupBy Object
Calling `.groupby()` splits the DataFrame into hidden groups based on values in the chosen column. It does not display a table immediately; it waits for your calculation instruction.
- **Syntax:** `groups = df.groupby("CategoryColumn")`
- **Example:**
  ```python
  df = pd.DataFrame({
      'Category1': ['A', 'B', 'A', 'B', 'A', 'B'],
      'Value1': [1, 2, 3, 4, 5, 6]
  })
  groups = df.groupby('Category1')
  ```

#### 2. Looking Inside the Groups
You can inspect what rows ended up inside each group in two ways:

- **Method A: Looping Through Every Group**
  ```python
  for key, group_df in groups:
      print(f"=== Group {key} ===")
      print(group_df)
  ```

- **Method B: Extracting One Specific Group (`.get_group()`)**
  Instead of looping through all categories, pull out only the rows belonging to one specific label:
  ```python
  # Pulls out only rows where Category1 == 'A'
  group_a = groups.get_group('A')
  print(group_a)
  # Output:
  #   Category1  Value1
  # 0         A       1
  # 2         A       3
  # 4         A       5
  ```

#### 3. Calculating Group Aggregations (`.mean()`, `.sum()`, `.count()`)
To calculate numbers per group, pick the numerical column you want to measure and apply the aggregation function:
- **Example:**
  ```python
  # Calculates the average of Value1 for each Category1
  avg_val = groups['Value1'].mean()
  print(avg_val)
  # Output:
  # Category1
  # A    3.0
  # B    4.0
  ```

#### 4. Grouping by Multiple Columns 
Pass a list of column names in brackets `['Category1', 'Category2']`. Pandas creates nested sub-categories:
- **Example:**
  ```python
  df = pd.DataFrame({
      'Category1': ['A', 'B', 'A', 'B', 'A', 'B'],
      'Category2': ['A', 'A', 'B', 'B', 'C', 'C'],
      'Value1': [1, 2, 3, 4, 5, 6]
  })
  # Groups first by Category1, then sub-groups by Category2
  multi_group = df.groupby(['Category1', 'Category2'])['Value1'].sum()
  print(multi_group)
  ```

#### 5. Grouping by a Custom Rule / Function 
You do not have to group only by pre-existing text columns. You can define your own rule using a function or inline `lambda` passed into the `by=` parameter:
- **Example:** Split rows into two buckets: "Small" (where Value1 < 3) and "Large" (where Value1 >= 3):
  ```python
  # Using a lambda function:
  val_groups = df.groupby(by=lambda idx: "Small" if df.loc[idx, "Value1"] < 3 else "Large")

  for label, subset in val_groups:
      print(f"Bucket: {label}")
      print(subset)
  ```

In [ ]:
# --- Demo: GroupBy Operations ---
import pandas as pd

# Sample data table
df = pd.DataFrame({
    'Category1': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Category2': ['A', 'A', 'B', 'B', 'C', 'C'],
    'Value1': [1, 2, 3, 4, 5, 6],
    'Value2': [5, 6, 7, 8, 9, 10]
})

print("=== Original Table ===")
print(df)

# 1. Group by Category1 and compute the mean of Value1
groups = df.groupby('Category1')
print("\n=== Mean of Value1 by Category1 ===")
print(groups['Value1'].mean())

# 2. Extract a single group with .get_group()
print("\n=== Rows where Category1 == 'A' ===")
print(groups.get_group('A'))

# 3. Multi-column grouping (Category1 and Category2)
print("\n=== Sum of Value1 & Value2 by Two Categories ===")
print(df.groupby(['Category1', 'Category2'])[['Value1', 'Value2']].sum())

# 4. Custom rule grouping with lambda
rule_groups = df.groupby(by=lambda idx: "Small" if df.loc[idx, "Value1"] < 3 else "Large")
print("\n=== Grouped by Custom Rule (Mean of Value2) ===")
print(rule_groups['Value2'].mean())

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 1: Department Sales Grouping and Analysis

**Goal:** Split an employee sales records table by teams, inspect isolated group buckets, and calculate group-level averages.

**Requirements:**
1. Create a DataFrame named `sales_df` with the following columns:
   - `'Department'`: `['Electronics', 'Clothing', 'Electronics', 'Clothing', 'Electronics', 'Clothing']`
   - `'Branch'`: `['North', 'North', 'South', 'South', 'North', 'South']`
   - `'Units'`: `[15, 25, 30, 10, 40, 20]`
   - `'Revenue'`: `[1500, 800, 3200, 400, 4100, 950]`
2. Group the data by `'Department'` and calculate the average `'Revenue'` for each department.
3. Use the `.get_group()` method to extract and display all rows belonging exclusively to the `'Electronics'` department.
4. Group by both `'Department'` and `'Branch'` simultaneously, calculating the total sum of `'Units'` and `'Revenue'`.
5. Group the table into two buckets—`'High Revenue'` for rows where `'Revenue' >= 2000` and `'Low Revenue'` otherwise—and display the count of records in each bucket.

<details>
<summary>💡 Need a hint?</summary>

- For step 2, remember: `df.groupby('Col')['TargetCol'].mean()`.
- For step 3, call `.get_group('GroupName')` on the created GroupBy object.
- For step 4, pass multiple column names in a list inside `.groupby([...])` and target multiple numeric columns using double brackets `[['Units', 'Revenue']]`.
- For step 5, pass a conditional `lambda idx: ...` into the `by=` parameter of `groupby()`.
</details>

In [ ]:
# --- Exercise 1: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 3.2 Concatenating DataFrames with `pd.concat`

### Concept Summary
Concatenation means **stacking** multiple DataFrames together. Think of it like taking printed sheets of paper and taping them together—either placing one page directly below another (vertical) or taping them side-by-side (horizontal).

The core function is:
```python
pd.concat(objs, axis=0, join='outer', ignore_index=False)
```

---

### Key Parameters Explained

#### 1. The List of Tables (`objs`)
You must pass the DataFrames bundled together inside a Python list or tuple: `[df1, df2]`.

#### 2. The Direction (`axis=0` vs `axis=1`)
- **`axis=0` (Vertical Stacking — Default):**
  - Stacks tables top-to-bottom (adds more rows).
  - Used when you receive new batches of records (like combining January sales with February sales).
- **`axis=1` (Horizontal Stacking):**
  - Places tables side-by-side (adds more columns).
  - Used when you have additional attributes for the same records.

#### 3. Resetting Row Numbers (`ignore_index=True`)
When you stack tables vertically, each table brings its original index numbers with it. This creates duplicate row numbers (`0, 1, 2, 0, 1, 2`).
- Setting `ignore_index=True` wipes out old row numbers and assigns a clean, continuous sequence: `0, 1, 2, 3, 4, 5...`.

#### 4. Handling Column Mismatches (`join='outer'` vs `join='inner'`)
What happens if Table 1 has columns `[name, age]` and Table 2 has `[name, city]`?
- **`join='outer'` (Default — Union):** Keeps **all** columns from both tables. Any cell where data didn't exist is filled with `NaN`.
- **`join='inner'` (Intersection):** Keeps **only columns present in BOTH tables**. Any non-matching column is discarded.

```text
=========================================================================================
                      VERTICAL CONCATENATION: OUTER VS. INNER JOIN
=========================================================================================

  Table 1:                     Table 2:
  +----+------+-----+          +----+------+--------+
  | id | name | age |          | id | city | salary |
  +----+------+-----+          +----+------+--------+

  --- join='outer' (Keep all 5 columns, fill missing with NaN) ---
  +----+------+-----+----------+--------+
  | id | name | age |   city   | salary |
  +----+------+-----+----------+--------+
  |  1 | Alice|  25 |   NaN    |  NaN   |  <- From Table 1
  |  3 | NaN  | NaN | New York | 50000  |  <- From Table 2
  +----+------+-----+----------+--------+

  --- join='inner' (Keep ONLY columns present in BOTH: 'id') ---
  +----+
  | id |
  +----+
  |  1 |
  |  3 |
  +----+
=========================================================================================
```

In [ ]:
# --- Demo: pd.concat (Vertical, Horizontal, Outer, Inner) ---
import pandas as pd

# Sample DataFrames
df1 = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'David'],
    'age': [25, 30, 35, 40]
})

df2 = pd.DataFrame({
    'id': [3, 4, 5, 6],
    'city': ['New York', 'San Francisco', 'London', 'Paris'],
    'salary': [50000, 60000, 55000, 65000]
})

df3 = pd.DataFrame({
    'name': ['Eve', 'Frank'],
    'age': [28, 32],
    'city': ['Berlin', 'Tokyo']
})

# 1. Vertical concatenation (outer join - keeps all columns)
print("=== Vertical Concatenation (Outer Join) ===")
print(pd.concat([df1, df3], axis=0, ignore_index=True))

# 2. Vertical concatenation (inner join - keeps only shared columns: 'name', 'age')
print("\n=== Vertical Concatenation (Inner Join) ===")
print(pd.concat([df1, df3], axis=0, join='inner', ignore_index=True))

# 3. Horizontal concatenation (axis=1 - side by side)
print("\n=== Horizontal Concatenation (axis=1) ===")
print(pd.concat([df1, df2], axis=1))

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 2: Regional Sales Stacking

**Goal:** Combine regional transaction batches vertically and horizontally, manage row index re-numbering, and control column retention with join rules.

**Requirements:**
1. Create two DataFrames:
   - `north_sales`: columns `'Store_ID'` (`[101, 102]`), `'Manager'` (`['Ava', 'Liam']`), and `'Sales'` (`[12000, 15000]`)
   - `south_sales`: columns `'Store_ID'` (`[103, 104]`), `'Manager'` (`['Noah', 'Mia']`), and `'Bonus'` (`[500, 700]`)
2. Stack `north_sales` and `south_sales` vertically using `pd.concat`:
   - Reset the row index cleanly so labels run sequentially from 0 to 3.
   - Keep all columns (outer join), and observe where `NaN` values appear.
3. Stack the same two DataFrames vertically again, but use an **inner join** so that only columns present in both tables are kept.
4. Create a small DataFrame `store_targets` with 2 rows and column `'Target'` (`[14000, 16000]`).
   - Concatenate `north_sales` and `store_targets` horizontally (`axis=1`) and display the result.

<details>
<summary>💡 Need a hint?</summary>

- For vertical stacking, pass `axis=0` and `ignore_index=True`.
- To retain only columns shared by both DataFrames, pass `join='inner'`.
- For horizontal combination, set `axis=1`.
</details>

In [ ]:
# --- Exercise 2: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 3.3 Merging DataFrames with `df.merge` (SQL Joins)

### Concept Summary
While `pd.concat` blindly pastes tables together by physical position, **`merge()` links tables together by matching common keys** (values like `id`, `user_name`, or `product_sku`). This functions exactly like SQL joins in relational databases.

**Core Syntax:**
```python
df1.merge(df2, on="shared_column", how="inner")
```

---

### Types of Joins (`how=...`) 

| Join Type | What it Does | Visual Venn Diagram | Real-World Example |
| :--- | :--- | :---: | :--- |
| **`'inner'`** (Default) | Keeps only rows where the key exists in **both** tables | ( Left ∩ Right ) | Customers who actually placed an order |
| **`'left'`** | Keeps **all rows from the left table**, matching right data where possible (`NaN` if no match) | ( Left ) | All enrolled students, plus exam scores if they showed up |
| **`'right'`** | Keeps **all rows from the right table**, matching left data where possible | ( Right ) | All store catalog items, plus buyer info if any sold |
| **`'outer'`** | Keeps **all rows from both tables** (union). Fills any missing data with `NaN` | ( Left ∪ Right ) | Master list combining two merged hospital registries |
| **`'cross'`** | Creates every possible pairing between rows (Cartesian product) | ( Left × Right ) | Combining 3 shirt sizes with 4 shirt colors (12 combinations) |

---

### Matching Column Names: `on` vs. `left_on` / `right_on` 

- **When both tables share the exact same column name:**
  ```python
  # Both df1 and df2 have a column called 'id'
  df1.merge(df2, on='id', how='inner')
  ```

- **When the linking column has different names in each table:**
  For example, `df1` calls it `'name'` but `df3` calls it `'first_name'`:
  ```python
  df1.merge(df3, left_on='name', right_on='first_name', how='outer')
  ```

- **When joining on row index labels:**
  If the key is stored in the row index instead of a column, pass `left_index=True` or `right_index=True`.

In [ ]:
# --- Demo: Merging DataFrames (Inner, Left, Outer, left_on / right_on) ---
import pandas as pd

# Customers table
df1 = pd.DataFrame({
    'id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'David'],
    'age': [25, 30, 35, 40]
})

# Orders table (IDs 1 & 2 have no orders; IDs 5 & 6 are new)
df2 = pd.DataFrame({
    'id': [3, 4, 5, 6],
    'city': ['New York', 'San Francisco', 'London', 'Paris'],
    'salary': [50000, 60000, 55000, 65000]
})

# Different header name table
df3 = pd.DataFrame({
    'first_name': ['Alice', 'Bob'],
    'city': ['Berlin', 'Tokyo'],
    'salary': [50000, 60000]
})

# 1. Inner Merge (only shared IDs: 3 and 4 appear)
print("=== Inner Merge (Common IDs 3 & 4 only) ===")
print(df1.merge(df2, on='id', how='inner'))

# 2. Left Merge (keeps all 4 customers from df1, NaNs for non-matching orders)
print("\n=== Left Merge (All df1 Customers Kept) ===")
print(df1.merge(df2, on='id', how='left'))

# 3. Outer Merge (keeps all records from both tables)
print("\n=== Outer Merge (All Records Kept) ===")
print(df1.merge(df2, on='id', how='outer'))

# 4. Merging with different column names (left_on & right_on)
print("\n=== Merging with left_on='name' and right_on='first_name' ===")
print(df1.merge(df3, left_on='name', right_on='first_name', how='inner'))

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 3: Relational Customer & Transaction Merges

**Goal:** Connect customer account profiles with transaction records using primary keys, evaluate inner vs left join consequences, and merge mismatched headers.

**Requirements:**
1. Create two DataFrames:
   - `users`: columns `'user_id'` (`[1, 2, 3, 4]`), `'name'` (`['Elena', 'Lucas', 'Sophie', 'Jacob']`), and `'tier'` (`['Gold', 'Silver', 'Bronze', 'Gold']`)
   - `purchases`: columns `'user_id'` (`[2, 3, 5]`), `'item'` (`['Laptop', 'Phone', 'Tablet']`), and `'cost'` (`[1200, 800, 600]`)
2. Perform an **inner merge** between `users` and `purchases` on `'user_id'`. Identify which users are kept in the output.
3. Perform a **left merge** keeping all registered users from `users`. Observe which values appear as `NaN`.
4. Perform a **full outer merge** between both tables to capture all users and any unlinked purchases.
5. Given a third DataFrame:
   ```python
   rewards = pd.DataFrame({
       'member_name': ['Elena', 'Lucas'],
       'points': [2500, 1400]
   })
   ```
   - Merge `users` with `rewards` so that `'name'` from `users` matches with `'member_name'` from `rewards` using an inner join.

<details>
<summary>💡 Need a hint?</summary>

- For step 2: `users.merge(purchases, on='user_id', how='inner')`.
- For step 3: Change `how='left'`.
- For step 4: Change `how='outer'`.
- For step 5: Use `left_on='name'` and `right_on='member_name'`.
</details>

In [ ]:
# --- Exercise 3: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>